In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
raw_car = "C:/Users/Lenovo/Documents/R programming/proccesing lsson3/raw_car_dataset.csv"
df = pd.read_csv(raw_car)

#### INITIAL SNAPSHOT


In [3]:
print("\n=== INITIAL HEAD ===")
print(df.head(10))


=== INITIAL HEAD ===
    Price  Odometer_km  Doors  Accidents Location  Year
0  $1,500     137830.0    4.0          1     City  1998
1  4171.0          NaN    4.0          0    Rural  2016
2  $5,331     107302.0    4.0          0   Suburb  2014
3  1500.0     141838.0    4.0          1   Suburb  1999
4  1500.0          NaN    3.0          0     City  2022
5  $1,500     211171.0    4.0          0       ??  2003
6  1500.0     222235.0    4.0          2    Rural  2004
7  $1,500     105068.0    5.0          1     City  2002
8  $2,291      90015.0    4.0          0    Rural  2007
9  1500.0     125976.0    2.0          0     City  2002


##### Shape of data

In [4]:
print(df.shape)

(145, 6)


##### INITIAL INFO 

In [5]:
print("\n=== INITIAL INFO ===")
print(df.info())


=== INITIAL INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Price        145 non-null    object 
 1   Odometer_km  138 non-null    float64
 2   Doors        138 non-null    float64
 3   Accidents    145 non-null    int64  
 4   Location     140 non-null    object 
 5   Year         145 non-null    int64  
dtypes: float64(2), int64(2), object(2)
memory usage: 6.9+ KB
None


##### Missing Count

In [6]:
print(df.isnull().sum())

Price          0
Odometer_km    7
Doors          7
Accidents      0
Location       5
Year           0
dtype: int64


##### Clean target formatting.

In [7]:
df["Price"] = df["Price"].replace(r"[\$,]", "", regex=True).astype(float)

print(df["Price"])

0      1500.0
1      4171.0
2      5331.0
3      1500.0
4      1500.0
        ...  
140    1500.0
141    1500.0
142    1500.0
143    1500.0
144    1500.0
Name: Price, Length: 145, dtype: float64


##### Fix categorical issues BEFORE imputation

In [8]:
df["Location"] = df["Location"].replace({"Subrb": "Suburb", "??": pd.NA})

print (df.info)

<bound method DataFrame.info of       Price  Odometer_km  Doors  Accidents Location  Year
0    1500.0     137830.0    4.0          1     City  1998
1    4171.0          NaN    4.0          0    Rural  2016
2    5331.0     107302.0    4.0          0   Suburb  2014
3    1500.0     141838.0    4.0          1   Suburb  1999
4    1500.0          NaN    3.0          0     City  2022
..      ...          ...    ...        ...      ...   ...
140  1500.0     223193.0    3.0          0     City  2003
141  1500.0     124567.0    NaN          2   Suburb  2002
142  1500.0     203153.0    4.0          0   Suburb  2004
143  1500.0     180030.0    4.0          1     City  2009
144  1500.0     211171.0    4.0          0     <NA>  2003

[145 rows x 6 columns]>


##### Convert unknowns (e.g., "??") to missing, then recount including missing.

In [10]:
df["Location"] = df["Location"].replace("??", np.nan)

print(df["Location"].value_counts(dropna=False))

City      59
Suburb    53
Rural     21
NaN       12
Name: Location, dtype: int64


### Impute missing values

##### Odometer_km → median; Doors/Accidents → mode; Location → mode.

In [11]:
df["Odometer_km"] = df["Odometer_km"].fillna(df["Odometer_km"].median())

print(df["Odometer_km"])

0      137830.0
1      128548.0
2      107302.0
3      141838.0
4      128548.0
         ...   
140    223193.0
141    124567.0
142    203153.0
143    180030.0
144    211171.0
Name: Odometer_km, Length: 145, dtype: float64


In [12]:
df["Doors"] = df["Doors"].fillna(df["Doors"].mode()[0])

print(df["Doors"])

0      4.0
1      4.0
2      4.0
3      4.0
4      3.0
      ... 
140    3.0
141    4.0
142    4.0
143    4.0
144    4.0
Name: Doors, Length: 145, dtype: float64


In [13]:
df["Accidents"] = df["Accidents"].fillna(df["Accidents"].mode()[0])

print(df["Accidents"])

0      1
1      0
2      0
3      1
4      0
      ..
140    0
141    2
142    0
143    1
144    0
Name: Accidents, Length: 145, dtype: int64


In [14]:
df["Location"] = df["Location"].fillna(df["Location"].mode()[0])

print(df["Location"])

0        City
1       Rural
2      Suburb
3      Suburb
4        City
        ...  
140      City
141    Suburb
142    Suburb
143      City
144      City
Name: Location, Length: 145, dtype: object


In [15]:
print(df.isnull().sum())

Price          0
Odometer_km    0
Doors          0
Accidents      0
Location       0
Year           0
dtype: int64


### Remove duplicates

In [16]:
before = df.shape
df = df.drop_duplicates()
after = df.shape
print(f"Dropped duplicates: {before} → {after}")

Dropped duplicates: (145, 6) → (140, 6)


### IQR capping

In [17]:
def iqr_fun(series, k=1.5):
    q1, q3= series.quantile([0.25, 0.75])
    iqr= q3 - q1
    lower= q1 - k * iqr
    upper= q3 + k * iqr
    return lower, upper


low_price, high_price = iqr_fun(df["Price"])

low_Odometer_km, high_Odometer_km= iqr_fun(df["Odometer_km"])

df["Price"] = df["Price"].clip(lower=low_price, upper=high_price)
df["Odometer_km"] = df["Odometer_km"].clip(lower=low_Odometer_km, upper=high_Odometer_km)

# print(f"PRICES: LOW {low_price} → HIGH {high_price}")

# print(f"ODOMETER: LOW {low_Odometer_km} → HIGH {high_Odometer_km}")

print("Working")





Working


#### One-Hot Encode Categorical(s)


In [18]:
print(type(df))

<class 'pandas.core.frame.DataFrame'>


In [19]:
print(df.head(10))

    Price  Odometer_km  Doors  Accidents Location  Year
0  1500.0     137830.0    4.0          1     City  1998
1  4171.0     128548.0    4.0          0    Rural  2016
2  5331.0     107302.0    4.0          0   Suburb  2014
3  1500.0     141838.0    4.0          1   Suburb  1999
4  1500.0     128548.0    3.0          0     City  2022
5  1500.0     211171.0    4.0          0     City  2003
6  1500.0     222235.0    4.0          2    Rural  2004
7  1500.0     105068.0    5.0          1     City  2002
8  2291.0      90015.0    4.0          0    Rural  2007
9  1500.0     125976.0    2.0          0     City  2002


In [30]:
print(df.columns.tolist())

['Price', 'Odometer_km', 'Doors', 'Accidents', 'Year', 'Location_Rural', 'Location_Suburb', 'Location_City_0', 'Location_City_1']


In [28]:
df = pd.get_dummies(df, columns=["Location_City"], drop_first=False, dtype=int)

# print(df)

print(df.head(10))

    Price  Odometer_km  Doors  Accidents  Year  Location_Rural  \
0  1500.0     137830.0    4.0          1  1998               0   
1  4171.0     128548.0    4.0          0  2016               1   
2  5331.0     107302.0    4.0          0  2014               0   
3  1500.0     141838.0    4.0          1  1999               0   
4  1500.0     128548.0    3.0          0  2022               0   
5  1500.0     211171.0    4.0          0  2003               0   
6  1500.0     222235.0    4.0          2  2004               1   
7  1500.0     105068.0    5.0          1  2002               0   
8  2291.0      90015.0    4.0          0  2007               1   
9  1500.0     125976.0    2.0          0  2002               0   

   Location_Suburb  Location_City_0  Location_City_1  
0                0                0                1  
1                0                1                0  
2                1                1                0  
3                1                1                0 

In [31]:
from datetime import datetime

# Get the current year
CURRENT_YEAR = datetime.now().year

df["Car_Age"] = CURRENT_YEAR - df["Year"]

print(df.head(10))

    Price  Odometer_km  Doors  Accidents  Year  Location_Rural  \
0  1500.0     137830.0    4.0          1  1998               0   
1  4171.0     128548.0    4.0          0  2016               1   
2  5331.0     107302.0    4.0          0  2014               0   
3  1500.0     141838.0    4.0          1  1999               0   
4  1500.0     128548.0    3.0          0  2022               0   
5  1500.0     211171.0    4.0          0  2003               0   
6  1500.0     222235.0    4.0          2  2004               1   
7  1500.0     105068.0    5.0          1  2002               0   
8  2291.0      90015.0    4.0          0  2007               1   
9  1500.0     125976.0    2.0          0  2002               0   

   Location_Suburb  Location_City_0  Location_City_1  Car_Age  
0                0                0                1       28  
1                0                1                0       10  
2                1                1                0       12  
3                

In [32]:
# Log Price

df["Log_Price"] = np.log1p(df["Price"])

print(df.head())

    Price  Odometer_km  Doors  Accidents  Year  Location_Rural  \
0  1500.0     137830.0    4.0          1  1998               0   
1  4171.0     128548.0    4.0          0  2016               1   
2  5331.0     107302.0    4.0          0  2014               0   
3  1500.0     141838.0    4.0          1  1999               0   
4  1500.0     128548.0    3.0          0  2022               0   

   Location_Suburb  Location_City_0  Location_City_1  Car_Age  Log_Price  
0                0                0                1       28   7.313887  
1                0                1                0       10   8.336151  
2                1                1                0       12   8.581482  
3                1                1                0       27   7.313887  
4                0                0                1        4   7.313887  


In [33]:
# Calculate KM per year safely

# print(df.head(10))

df['Km_Per_Year'] = np.where(
    df["Car_Age"] > 0,
    df["Odometer_km"] / df["Car_Age"],
    df["Odometer_km"]
).astype(float).round(2)

# print(df.info())

df[["Odometer_km", "Car_Age", "Km_Per_Year"]]

,Odometer_km,Car_Age,Km_Per_Year
0,137830.0,28,4922.50
1,128548.0,10,12854.80
2,107302.0,12,8941.83
3,141838.0,27,5253.26
4,128548.0,4,32137.00
...,...,...,...
135,170466.0,12,14205.50
136,219401.0,14,15671.50
137,223338.0,3,74446.00
138,168376.0,18,9354.22


In [38]:
df["Location_Rural"] = df["Location_City_0"].astype(int)

df.head(10)

,Price,Odometer_km,Doors,Accidents,Year,Location_Rural,Location_Suburb,Location_City_0,Location_City_1,Car_Age,Log_Price,Km_Per_Year
0,1500.0,137830.0,4.0,1,1998,0,0,0,1,28,7.313887,4922.50
1,4171.0,128548.0,4.0,0,2016,1,0,1,0,10,8.336151,12854.80
2,5331.0,107302.0,4.0,0,2014,1,1,1,0,12,8.581482,8941.83
3,1500.0,141838.0,4.0,1,1999,1,1,1,0,27,7.313887,5253.26
4,1500.0,128548.0,3.0,0,2022,0,0,0,1,4,7.313887,32137.00
5,1500.0,211171.0,4.0,0,2003,0,0,0,1,23,7.313887,9181.35
6,1500.0,222235.0,4.0,2,2004,1,0,1,0,22,7.313887,10101.59
7,1500.0,105068.0,5.0,1,2002,0,0,0,1,24,7.313887,4377.83
8,2291.0,90015.0,4.0,0,2007,1,0,1,0,19,7.737180,4737.63
9,1500.0,125976.0,2.0,0,2002,0,0,0,1,24,7.313887,5249.00


### Feature Scaling (X only)

In [40]:
dont_scale = {"Price", "LogPrice"}

numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.to_list()

exclude = [c for c in df.columns if c.startswith("Location_")] + ["Is_City"]
num_features_to_scale = [c for c in numeric_cols if c not in dont_scale and c not in exclude]

print(df.head(10))

    Price  Odometer_km  Doors  Accidents  Year  Location_Rural  \
0  1500.0     137830.0    4.0          1  1998               0   
1  4171.0     128548.0    4.0          0  2016               1   
2  5331.0     107302.0    4.0          0  2014               1   
3  1500.0     141838.0    4.0          1  1999               1   
4  1500.0     128548.0    3.0          0  2022               0   
5  1500.0     211171.0    4.0          0  2003               0   
6  1500.0     222235.0    4.0          2  2004               1   
7  1500.0     105068.0    5.0          1  2002               0   
8  2291.0      90015.0    4.0          0  2007               1   
9  1500.0     125976.0    2.0          0  2002               0   

   Location_Suburb  Location_City_0  Location_City_1  Car_Age  Log_Price  \
0                0                0                1       28   7.313887   
1                0                1                0       10   8.336151   
2                1                1          

In [41]:
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.to_list()


# print(numeric_cols)


dont_use = {"Price", "Log_Price", "Is_Urban"}

locs = {"Location_"}

num_to_sclale = []

for f in numeric_cols:
    if (f in dont_use or f.startswith("Location_")):
        continue
    num_to_sclale.append(f)

print(num_to_sclale)

scaler = StandardScaler()

df[num_to_sclale] = scaler.fit_transform(df[num_to_sclale])

print(df.head(10))

['Odometer_km', 'Doors', 'Accidents', 'Year', 'Car_Age', 'Km_Per_Year']
    Price  Odometer_km     Doors  Accidents      Year  Location_Rural  \
0  1500.0     0.128390  0.254091   0.316968 -1.686714               0   
1  4171.0    -0.044709  0.254091  -0.820867  0.794617               1   
2  5331.0    -0.440923  0.254091  -0.820867  0.518913               1   
3  1500.0     0.203135  0.254091   0.316968 -1.548862               1   
4  1500.0    -0.044709 -0.931668  -0.820867  1.621727               0   
5  1500.0     1.496119  0.254091  -0.820867 -0.997456               0   
6  1500.0     1.702451  0.254091   1.454804 -0.859604               1   
7  1500.0    -0.482585  1.439851   0.316968 -1.135307               0   
8  2291.0    -0.763307  0.254091  -0.820867 -0.446049               1   
9  1500.0    -0.092674 -2.117428  -0.820867 -1.135307               0   

   Location_Suburb  Location_City_0  Location_City_1   Car_Age  Log_Price  \
0                0                0            

In [42]:
df.head(10)

,Price,Odometer_km,Doors,Accidents,Year,Location_Rural,Location_Suburb,Location_City_0,Location_City_1,Car_Age,Log_Price,Km_Per_Year
0,1500.0,0.128390,0.254091,0.316968,-1.686714,0,0,0,1,1.686714,7.313887,-0.615631
1,4171.0,-0.044709,0.254091,-0.820867,0.794617,1,0,1,0,-0.794617,8.336151,0.070446
2,5331.0,-0.440923,0.254091,-0.820867,0.518913,1,1,1,0,-0.518913,8.581482,-0.267993
3,1500.0,0.203135,0.254091,0.316968,-1.548862,1,1,1,0,1.548862,7.313887,-0.587023
4,1500.0,-0.044709,-0.931668,-0.820867,1.621727,0,0,0,1,-1.621727,7.313887,1.738196
5,1500.0,1.496119,0.254091,-0.820867,-0.997456,0,0,0,1,0.997456,7.313887,-0.247276
6,1500.0,1.702451,0.254091,1.454804,-0.859604,1,0,1,0,0.859604,7.313887,-0.167683
7,1500.0,-0.482585,1.439851,0.316968,-1.135307,0,0,0,1,1.135307,7.313887,-0.662741
8,2291.0,-0.763307,0.254091,-0.820867,-0.446049,1,0,1,0,0.446049,7.737180,-0.631621
9,1500.0,-0.092674,-2.117428,-0.820867,-1.135307,0,0,0,1,1.135307,7.313887,-0.587392


In [43]:
df.info()

print(df.shape)

print(df.describe())

print("NULL VALUE COUNTS IN DATAFRAME")

print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 140 entries, 0 to 139
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Price            140 non-null    float64
 1   Odometer_km      140 non-null    float64
 2   Doors            140 non-null    float64
 3   Accidents        140 non-null    float64
 4   Year             140 non-null    float64
 5   Location_Rural   140 non-null    int32  
 6   Location_Suburb  140 non-null    int32  
 7   Location_City_0  140 non-null    int32  
 8   Location_City_1  140 non-null    int32  
 9   Car_Age          140 non-null    float64
 10  Log_Price        140 non-null    float64
 11  Km_Per_Year      140 non-null    float64
dtypes: float64(8), int32(4)
memory usage: 12.0 KB
(140, 12)
             Price   Odometer_km         Doors     Accidents          Year  \
count   140.000000  1.400000e+02  1.400000e+02  1.400000e+02  1.400000e+02   
mean   3175.456250 -1.982541e-18  

In [44]:
OUT_PATH = "car_l3_clean_ready.csv"

df.to_csv(OUT_PATH)

In [45]:
print(pd.__version__)
print(np.__version__)
import sklearn
print(sklearn.__version__)

1.3.4
1.20.3
0.24.2
